# House Prices — Data Preparation
The split precedes fitting. Only the training fold determines imputation, scaling, and categories.

In [1]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import numpy as np
from src.data import read_data, split_data
from src.preprocess import prepare_data
from src.dataset import make_loaders
train, test = read_data()
split = split_data(train, test, seed=42)
prepared = prepare_data(split)
loaders = make_loaders(prepared, batch_size=32, seed=42)
print('Train/val/test:', prepared.x_train.shape, prepared.x_val.shape, prepared.x_test.shape)
print('X dtype, y dtype:', prepared.x_train.dtype, prepared.y_train.dtype)
print('y shape:', prepared.y_train.shape, prepared.y_val.shape)
print('Dense MiB:', [round(x.nbytes / 2**20, 2) for x in [prepared.x_train, prepared.x_val, prepared.x_test]])
print('Finite:', [np.isfinite(x).all() for x in [prepared.x_train, prepared.x_val, prepared.x_test]])
print('Loader batches:', [len(loader) for loader in loaders])

Train/val/test: (1168, 286) (292, 286) (1459, 286)
X dtype, y dtype: float32 float32
y shape: (1168, 1) (292, 1)
Dense MiB: [1.27, 0.32, 1.59]
Finite: [np.True_, np.True_, np.True_]
Loader batches: [37, 10, 46, 37]


## Leakage control
`prepare_data` fits the `ColumnTransformer` on `split.x_train` only, then transforms validation and test. The `Id` and target columns never enter the feature matrix. Inference reuses the serialized transformer and feature order.